# Abhishek Lunagariya (MDS202402)
## Assignment 5


### Step 1: Download the Custom Image Dataset
To build our dataset, we will automatically download 100 images of ducks and 100 images of chickens using `bing-image-downloader`.

In [1]:
!pip install bing-image-downloader -q

from bing_image_downloader import downloader
import os
import shutil

# Create a base directory for the dataset
dataset_dir = 'dataset'
os.makedirs(dataset_dir, exist_ok=True)

print("Downloading Duck images...")
downloader.download("duck bird", limit=100, output_dir=dataset_dir, adult_filter_off=True, force_replace=False, timeout=60, verbose=False)

print("\nDownloading Chicken images...")
downloader.download("chicken bird", limit=100, output_dir=dataset_dir, adult_filter_off=True, force_replace=False, timeout=60, verbose=False)

# Rename folders for easier PyTorch ImageFolder handling
try:
    os.rename(os.path.join(dataset_dir, 'duck bird'), os.path.join(dataset_dir, 'duck'))
    os.rename(os.path.join(dataset_dir, 'chicken bird'), os.path.join(dataset_dir, 'chicken'))
except FileNotFoundError:
    pass # Folders might already be renamed if run twice

print("\nVerifying downloads:")
print(f"Ducks downloaded: {len(os.listdir(os.path.join(dataset_dir, 'duck')))}")
print(f"Chickens downloaded: {len(os.listdir(os.path.join(dataset_dir, 'chicken')))}")

[%] Downloading Images to /content/dataset/duck bird
[!] Issue getting: https://static.vecteezy.com/system/resources/previews/013/416/205/large_2x/a-beautiful-aquatic-wild-bird-duck-with-a-beak-and-wings-floats-against-the-background-of-water-in-the-river-lake-pond-sea-and-green-water-lilies-photo.jpg
[!] Error:: HTTP Error 403: Forbidden
[!] Issue getting: https://birdsandwetlands.com/wp-content/webp-express/webp-images/uploads/2023/04/Diving-Ducks-in-the-US.jpg.webp
[!] Error:: HTTP Error 403: Forbidden


[%] Done. Downloaded 100 images.

[%] Downloading Images to /content/dataset/chicken bird
[!] Issue getting: https://wallpapercrafter.com/th8007/1824755-chicken-breed-chicken-bird-agriculture-bill-poultry.jpg
[!] Error:: HTTP Error 403: Forbidden
[!] Issue getting: https://wallpapercrafter.com/desktop7/1824764-hen-hen-brahma-brown-nature-animals-image-animal.jpg
[!] Error:: HTTP Error 403: Forbidden
[!] Issue getting: https://media1.thehungryjpeg.com/thumbs/800_4403367_jr6g5v6p6lcdo

In [13]:
from PIL import Image
import os

dataset_dir = 'dataset'
corrupt_count = 0

print("Scanning for corrupt images...")
for folder in ['duck', 'chicken']:
    folder_path = os.path.join(dataset_dir, folder)
    if not os.path.exists(folder_path): continue

    for filename in os.listdir(folder_path):
        filepath = os.path.join(folder_path, filename)
        try:
            # PIL verify() checks the file header without loading the whole image into memory
            with Image.open(filepath) as img:
                img.verify()
        except Exception as e:
            print(f"Removing corrupted or unreadable image: {filepath}")
            os.remove(filepath)
            corrupt_count += 1

print(f"\nVerification complete. Removed {corrupt_count} corrupt images.")

Scanning for corrupt images...

Verification complete. Removed 0 corrupt images.


### Step 2: Data Preprocessing and Loading
Pre-trained models expect images in a specific format. Here, we resize all images to 224x224 pixels, convert them to PyTorch tensors, normalize them using standard ImageNet mean and standard deviation, and split our 200 images into an 80% training set and a 20% testing set.

In [14]:
import torch
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset

# 1. Define separate transformations for Train and Test
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(), # Augmentation
    transforms.RandomRotation(10),     # Augmentation
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),             # No augmentation for testing
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 2. Load the dataset twice, once with each transform pipeline
dataset_dir = 'dataset'
train_dataset_full = datasets.ImageFolder(root=dataset_dir, transform=train_transforms)
test_dataset_full = datasets.ImageFolder(root=dataset_dir, transform=test_transforms)

# 3. Create indices for an 80/20 split
num_samples = len(train_dataset_full)
torch.manual_seed(42)
indices = torch.randperm(num_samples).tolist()
train_size = int(0.8 * num_samples)

# 4. Apply indices to create Subsets (ensuring Train gets augmentations, Test doesn't)
train_dataset = Subset(train_dataset_full, indices[:train_size])
test_dataset = Subset(test_dataset_full, indices[train_size:])

# 5. Create DataLoaders
batch_size = 16
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

class_names = train_dataset_full.classes

print(f"Classes found: {class_names}")
print(f"Total valid images: {num_samples}")
print(f"Training images: {len(train_dataset)}")
print(f"Testing images: {len(test_dataset)}")

Classes found: ['chicken', 'duck']
Total valid images: 200
Training images: 160
Testing images: 40


### Step 3: Model Setup and Training
We will use **ResNet18**, a pre-trained Convolutional Neural Network. To prevent overfitting on our small dataset, we use Transfer Learning: we freeze the pre-trained feature extraction layers and only train a new, final classification layer to distinguish between our two classes (duck vs. chicken).

In [15]:
import torch.nn as nn
import torch.optim as optim
from torchvision import models
import time

# Check if GPU is available
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 1. Load pre-trained ResNet18
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# 2. Freeze the early layers (feature extractor)
for param in model.parameters():
    param.requires_grad = False

# 3. Replace the final layer to output 2 classes instead of 1000
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 2)
model = model.to(device)

# 4. Define Loss Function and Optimizer
criterion = nn.CrossEntropyLoss()
# We only optimize the parameters of the newly added final layer
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

# 5. Training Loop
num_epochs = 5
print("\nStarting training...")
start_time = time.time()

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    running_corrects = 0

    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        running_corrects += torch.sum(preds == labels.data)

    epoch_loss = running_loss / len(train_dataset)
    epoch_acc = running_corrects.double() / len(train_dataset)

    print(f'Epoch {epoch+1}/{num_epochs} - Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

time_elapsed = time.time() - start_time
print(f'\nTraining complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')

Using device: cuda:0

Starting training...
Epoch 1/5 - Loss: 0.6516 Acc: 0.6125
Epoch 2/5 - Loss: 0.4400 Acc: 0.8188
Epoch 3/5 - Loss: 0.2918 Acc: 0.9313
Epoch 4/5 - Loss: 0.2924 Acc: 0.8938
Epoch 5/5 - Loss: 0.2237 Acc: 0.9625

Training complete in 0m 41s


### Step 4: Evaluation and Classification Report
To ensure our model generalizes well to unseen data, we evaluate it on our test dataset. We then use `scikit-learn` to generate a comprehensive classification report showing Precision, Recall, and F1-Score for both the duck and chicken classes.

In [16]:
from sklearn.metrics import classification_report
import warnings
import torch

# Ignore minor sklearn warnings
warnings.filterwarnings('ignore')

model.eval() # Set model to evaluation mode
all_preds = []
all_labels = []

print("Evaluating model on test data...")

# Disable gradient calculation for inference
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)

        # Keep as tensors on the GPU and append to the list
        all_preds.append(preds)
        all_labels.append(labels)

# Concatenate all batches into single tensors, THEN move to CPU once
all_preds = torch.cat(all_preds).cpu().numpy()
all_labels = torch.cat(all_labels).cpu().numpy()

# Generate and print the classification report
print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=class_names))

Evaluating model on test data...

Classification Report:
              precision    recall  f1-score   support

     chicken       1.00      0.89      0.94        19
        duck       0.91      1.00      0.95        21

    accuracy                           0.95        40
   macro avg       0.96      0.95      0.95        40
weighted avg       0.95      0.95      0.95        40



### Part 2: Sentiment Analysis with Transformers
**Step 1: Unzip and Load the Dataset**
We will extract our manually uploaded `archive.zip` file and load the `train.csv` file into a Pandas DataFrame. We use `latin-1` encoding, which is standard for Twitter/social media datasets to prevent character reading errors, and then filter for the text and sentiment columns.

In [6]:
import pandas as pd
import zipfile
import os

zip_path = 'archive.zip'
extract_dir = 'sentiment_data'

# 1. Extract the zip file
print("Extracting archive.zip...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print("Files found in zip:", os.listdir(extract_dir))

# 2. Load the train.csv file
file_path = f"{extract_dir}/train.csv"
df = pd.read_csv(file_path, encoding='latin-1')

print("\nColumns in the dataset:", df.columns.tolist())

# 3. Filter for text and sentiment, and drop empty rows
try:
    df = df[['text', 'sentiment']].dropna()
    print(f"\nDataset loaded successfully! Total valid rows: {len(df)}")
    print("\nFirst 5 rows:")
    print(df.head())
    print("\nSentiment distribution:")
    print(df['sentiment'].value_counts())
except KeyError as e:
    print(f"\nKeyError: {e}")
    print("The column names don't exactly match 'text' and 'sentiment'. Paste this output so we can map them correctly!")

Extracting archive.zip...
Files found in zip: ['test.csv', 'train.csv', 'testdata.manual.2009.06.14.csv', 'training.1600000.processed.noemoticon.csv']

Columns in the dataset: ['textID', 'text', 'selected_text', 'sentiment', 'Time of Tweet', 'Age of User', 'Country', 'Population -2020', 'Land Area (Km²)', 'Density (P/Km²)']

Dataset loaded successfully! Total valid rows: 27480

First 5 rows:
                                                text sentiment
0                I`d have responded, if I were going   neutral
1      Sooo SAD I will miss you here in San Diego!!!  negative
2                          my boss is bullying me...  negative
3                     what interview! leave me alone  negative
4   Sons of ****, why couldn`t they put them on t...  negative

Sentiment distribution:
sentiment
neutral     11117
positive     8582
negative     7781
Name: count, dtype: int64


### Step 2: Preprocessing, Tokenization, and Dataset Creation
Social media text is highly noisy. We first use regular expressions to strip out URLs and @ user mentions to create a cleaner feature space for the model. Then, we map our labels, tokenize the full dataset using `DistilBertTokenizerFast`, and create our PyTorch DataLoaders.

In [17]:
!pip install transformers -q

import torch
import re
from transformers import DistilBertTokenizerFast
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader

# --- IMPROVEMENT 1: Text Preprocessing ---
def clean_text(text):
    text = str(text)
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    # Remove @ mentions and hashtags
    text = re.sub(r'\@\w+|\#', '', text)
    return text

print("Cleaning text data...")
df['text'] = df['text'].apply(clean_text)

# --- IMPROVEMENT 2: Use Full Dataset ---
# We use the full dataframe (df) instead of a sample.
# 1. Map string labels to integers
label_map = {'negative': 0, 'neutral': 1, 'positive': 2}
df['label'] = df['sentiment'].map(label_map)

# 2. Split dataset into training (80%) and testing (20%)
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['text'].tolist(), df['label'].tolist(), test_size=0.2, random_state=42
)

# 3. Load the pre-trained DistilBERT Tokenizer
print("Downloading DistilBERT tokenizer...")
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

# 4. Tokenize the text
print("Tokenizing text...")
train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=128)
test_encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=128)

# 5. Create a custom PyTorch Dataset
class SentimentDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

# Convert encodings to dataset objects
train_dataset = SentimentDataset(train_encodings, train_labels)
test_dataset = SentimentDataset(test_encodings, test_labels)

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

print("\nTokenization complete!")
print(f"Training dataset size: {len(train_dataset)}")
print(f"Testing dataset size: {len(test_dataset)}")

Cleaning text data...
Tokenizing text...

Tokenization complete!
Training dataset size: 21984
Testing dataset size: 5496


### Step 3: Model Training with LR Scheduler and Evaluation
We load `DistilBertForSequenceClassification`. To ensure stable fine-tuning on the full dataset, we implement a linear learning rate scheduler with a warmup period. This prevents catastrophic forgetting of pre-trained weights during the early phases of training. After 3 epochs, we evaluate the model on our test set.

In [18]:
from transformers import DistilBertForSequenceClassification, get_linear_schedule_with_warmup
import torch.optim as optim
from sklearn.metrics import classification_report
import time
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("Loading pre-trained DistilBERT model...")
model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=3)
model.to(device)

optimizer = optim.AdamW(model.parameters(), lr=5e-5)
num_epochs = 3

# --- IMPROVEMENT 3: Learning Rate Scheduler ---
total_steps = len(train_loader) * num_epochs
# Warmup for 10% of total training steps
num_warmup_steps = int(0.1 * total_steps)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=num_warmup_steps, num_training_steps=total_steps)

print(f"\nStarting training for {num_epochs} epochs on {device} (Full Dataset)...")
start_time = time.time()

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for batch in train_loader:
        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss

        loss.backward()
        optimizer.step()
        scheduler.step() # Update learning rate schedule

        running_loss += loss.item()

    print(f"Epoch {epoch+1}/{num_epochs} - Loss: {running_loss/len(train_loader):.4f}")

time_elapsed = time.time() - start_time
print(f'\nTraining complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')

# --- EVALUATION ---
print("\nEvaluating model and generating classification report...")
model.eval()
all_preds = []
all_labels = []

# Disable gradient calculation for inference
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Generate final report
target_names = ['negative', 'neutral', 'positive']
print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=target_names))

Loading pre-trained DistilBERT model...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Starting training for 3 epochs on cuda (Full Dataset)...
Epoch 1/3 - Loss: 0.6264
Epoch 2/3 - Loss: 0.4071
Epoch 3/3 - Loss: 0.2283

Training complete in 11m 30s

Evaluating model and generating classification report...

Classification Report:
              precision    recall  f1-score   support

    negative       0.79      0.79      0.79      1572
     neutral       0.76      0.76      0.76      2236
    positive       0.83      0.84      0.83      1688

    accuracy                           0.79      5496
   macro avg       0.79      0.79      0.79      5496
weighted avg       0.79      0.79      0.79      5496



### Final Evaluation and Conclusion
By training on the full 27,000+ row dataset and utilizing a linear learning rate scheduler with a warmup period, our fine-tuned DistilBERT model achieved a final accuracy of **79%** in just under 12 minutes of training.

**Key Takeaways:**
* **Performance:** The model is particularly strong at identifying positive sentiments, achieving an F1-score of 0.83 for that class.
* **Architecture:** The `UNEXPECTED` and `MISSING` weight notifications during model loading confirm that the pre-trained DistilBERT language modeling head was successfully discarded and replaced with our newly initialized 3-class sequence classification head, which was then trained from scratch on our specific Twitter dataset.
* **Optimization:** Stripping URLs and user mentions from the raw text provided a cleaner feature space, allowing the transformer's attention mechanisms to focus entirely on the linguistic context of the tweets.